HEALTH DATA ANALYTICS 


Identification: Country_code, country_name, region, income_level, year
Demographics: population, population_growth, urban_pct, fertility_rate, adolescent_fertility
Health Outcomes: life_expectancy, death_rate, birth_rate, under5_mortality, neonatal_mortality, maternal_mortality
Disease & Risk Factors: tb_incidence, hiv_incidence, obesity_pct, smoking_male, smoking_female, alcohol_per_capita, diabetes_pct
Infrastructure & Care: health_expenditure_pct_gdp, physicians_per_1000, hospital_beds_per_1000, immunization_measles, prenatal_care_pct
Economy & Poverty: gdp_per_capita, gdp_per_capita_ppp, poverty_rate 

DATA PREPARATION

In [102]:
%pip install pandas numpy matplotlib seaborn plotly scipy geopandas statsmodels nbformat --upgrade


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [103]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.express as px
import statistics as stats
from scipy import stats as st
import geopandas as gpd


In [104]:
df = pd.read_csv('health_indicators.csv')
df = pd.read_csv('indicator_metadata.csv')
df = pd.read_csv('country_latest.csv')

df.head()
df.columns
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 7802 entries, 0 to 7801
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_code    7802 non-null   str    
 1   country_name    7802 non-null   str    
 2   region          7802 non-null   str    
 3   income_level    7802 non-null   str    
 4   indicator_code  7802 non-null   str    
 5   indicator_name  7802 non-null   str    
 6   value           7802 non-null   float64
 7   year            7802 non-null   int64  
dtypes: float64(1), int64(1), str(6)
memory usage: 1.2 MB


In [105]:
df = pd.read_csv('indicator_metadata.csv')
df.head()
df.columns
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 43 entries, 0 to 42
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   indicator_code  43 non-null     str  
 1   indicator_name  43 non-null     str  
 2   category        43 non-null     str  
 3   unit            43 non-null     str  
 4   description     43 non-null     str  
dtypes: str(5)
memory usage: 6.8 KB


In [106]:
df = pd.read_csv('health_indicators.csv')
df.head()
df.columns
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 5275 entries, 0 to 5274
Data columns (total 48 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   country_code                   5275 non-null   str    
 1   country_name                   5275 non-null   str    
 2   region                         5275 non-null   str    
 3   income_level                   5275 non-null   str    
 4   year                           5275 non-null   int64  
 5   population                     5275 non-null   float64
 6   population_growth              5274 non-null   float64
 7   urban_pct                      5275 non-null   float64
 8   life_expectancy                5275 non-null   float64
 9   life_expectancy_male           5275 non-null   float64
 10  life_expectancy_female         5275 non-null   float64
 11  death_rate                     5275 non-null   float64
 12  birth_rate                     5275 non-null   float64
 13 

In [107]:
df = pd.read_csv('country_latest.csv')
missing_values = df.isnull().sum()
print(missing_values)


country_code      0
country_name      0
region            0
income_level      0
indicator_code    0
indicator_name    0
value             0
year              0
dtype: int64


In [108]:
df = pd.read_csv('health_indicators.csv')
missing_values = df.isnull().sum()
print(missing_values)

country_code                        0
country_name                        0
region                              0
income_level                        0
year                                0
population                          0
population_growth                   1
urban_pct                           0
life_expectancy                     0
life_expectancy_male                0
life_expectancy_female              0
death_rate                          0
birth_rate                          0
health_expenditure_pct_gdp        761
health_expenditure_per_capita     764
physicians_per_1000              2444
hospital_beds_per_1000           2377
nurses_per_1000                  2443
under5_mortality                  425
neonatal_mortality                425
maternal_mortality                667
tb_incidence                      218
hiv_incidence                    1661
communicable_death_pct           4177
noncommunicable_death_pct        4177
smoking_male                     3971
smoking_fema

In [109]:
df = pd.read_csv('indicator_metadata.csv')
missing_values = df.isnull().sum()
print(missing_values)

indicator_code    0
indicator_name    0
category          0
unit              0
description       0
dtype: int64


DATA CLEANING

In [110]:
df = pd.read_csv("health_indicators.csv")

df = df.sort_values(['country_name', 'year'])

numeric_cols = df.select_dtypes(include='number').columns

for col in numeric_cols:
    df[col] = (
        df.groupby('country_name')[col]
          .transform(
              lambda x: x.interpolate()
                         .ffill()
                         .bfill()
          )
    )

# Fill any remaining gaps with regional median
for col in numeric_cols:
    df[col] = (
        df.groupby('region')[col]
          .transform(lambda x: x.fillna(x.median()))
    )

print(df.isnull().sum())


country_code                        0
country_name                        0
region                              0
income_level                        0
year                                0
population                          0
population_growth                   0
urban_pct                           0
life_expectancy                     0
life_expectancy_male                0
life_expectancy_female              0
death_rate                          0
birth_rate                          0
health_expenditure_pct_gdp          0
health_expenditure_per_capita       0
physicians_per_1000                 0
hospital_beds_per_1000              0
nurses_per_1000                     0
under5_mortality                    0
neonatal_mortality                  0
maternal_mortality                  0
tb_incidence                        0
hiv_incidence                       0
communicable_death_pct              0
noncommunicable_death_pct           0
smoking_male                        0
smoking_fema

DATA VISUALISATION

In [111]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("health_indicators.csv")

# Get latest year
latest_year = df["year"].max()

# Top 5 countries by life expectancy in latest year
top_countries = (
    df[df["year"] == latest_year]
    .nlargest(5, "life_expectancy")["country_name"]
)

# Historical data for those countries
plot_data = df[df["country_name"].isin(top_countries)]

# Create line chart
fig = px.line(
    plot_data,
    x="year",
    y="life_expectancy",
    color="country_name",
    markers=True,
    title="Top 5 Countries by Life Expectancy"
)

# Add life expectancy value at end of each line
for country in plot_data["country_name"].unique():

    country_df = (
        plot_data[plot_data["country_name"] == country]
        .sort_values("year")
    )

    last_row = country_df.iloc[-1]

    fig.add_annotation(
        x=last_row["year"],
        y=last_row["life_expectancy"],
        text=f"{last_row['life_expectancy']:.1f}",
        showarrow=False,
        xshift=20,
        font=dict(size=12, color="black")
    )

fig.update_layout(
    template="plotly_white",
    xaxis_title="Year",
    yaxis_title="Life Expectancy (Years)",
    width=900,
    height=700,
    margin=dict(r=80)
)

fig.show()

In [112]:


df = pd.read_csv("health_indicators.csv")

# Remove rows with missing values
plot_data = df.dropna(subset=["maternal_mortality", "poverty_rate"])

fig = px.scatter(
    plot_data,
    x="poverty_rate",
    y="maternal_mortality",
    color="region",
    hover_name="country_name",
    size="population",
    title="Maternal Mortality vs Poverty Rate"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Poverty Rate (%)",
    yaxis_title="Maternal Mortality",
    width=800,
    height=600
)

fig.show()

In [113]:
df = pd.read_csv("health_indicators.csv")

fig = px.scatter(
    long_df,
    x="Coverage",
    y="neonatal_mortality",
    color="Vaccine",
    facet_col="Vaccine",
    facet_col_wrap=3,
    trendline="ols",
    title="Neonatal Mortality vs Immunization Coverage"
)

fig.update_layout(
    template="plotly_white",
    width=1600,
    height=900
)

fig.show()

In [114]:
latest_year = df["year"].max()

top10 = (
    df[df["year"] == latest_year]
    .nlargest(10, "life_expectancy")
)

fig = px.bar(
    top10,
    x="country_name",
    y=["life_expectancy", "death_rate"],
    barmode="group",
    title="Top 10 Countries: Life Expectancy vs Death Rate"
)

fig.show()

In [115]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

country = "Kenya"

country_df = df[df["country_name"] == country]

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=country_df["year"],
        y=country_df["life_expectancy"],
        name="Life Expectancy"
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=country_df["year"],
        y=country_df["death_rate"],
        name="Death Rate"
    ),
    secondary_y=True,
)

fig.update_layout(
    title=f"{country}: Life Expectancy vs Death Rate"
)

fig.update_yaxes(title_text="Life Expectancy", secondary_y=False)
fig.update_yaxes(title_text="Death Rate", secondary_y=True)

fig.update_layout(width=800, height=600)
fig.show()

In [116]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("health_indicators.csv")

latest_year = df["year"].max()

plot_data = df[df["year"] == latest_year]

fig = px.scatter(
    plot_data,
    x="death_rate",
    y="life_expectancy",
    color="income_level",
    hover_name="country_name",
    size="population",
    title=f"Life Expectancy vs Death Rate ({latest_year})"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Death Rate",
    yaxis_title="Life Expectancy (Years)"
)

fig.show()

In [117]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("health_indicators.csv")

latest_year = df["year"].max()

plot_data = df[
    (df["year"] == latest_year)
].dropna(
    subset=["fertility_rate", "population_growth"]
)

fig = px.scatter(
    plot_data,
    x="fertility_rate",
    y="population_growth",
    trendline="ols",
    hover_name="country_name",
    title=f"Fertility Rate vs Population Growth ({latest_year})"
)

fig.update_layout(
    template="plotly_white",
    width=800,
    height=700
)

fig.show()

In [118]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("health_indicators.csv")

# Remove missing values
plot_data = df.dropna(
    subset=["health_expenditure_per_capita"]
)

# Average by region and year
region_trend = (
    plot_data
    .groupby(["year", "region"])["health_expenditure_per_capita"]
    .mean()
    .reset_index()
)

fig = px.line(
    region_trend,
    x="year",
    y="health_expenditure_per_capita",
    color="region",
    markers=True,
    title="Average Health Expenditure Per Capita by Region"
)

fig.update_layout(
    template="plotly_white",
    
    xaxis_title="Year",
    yaxis_title="Health Expenditure Per Capita ($)"
)

fig.show()

In [119]:
region_trend = (
    df.dropna(subset=["health_expenditure_pct_gdp"])
      .groupby(["year", "region"])["health_expenditure_pct_gdp"]
      .mean()
      .reset_index()
)

fig = px.line(
    region_trend,
    x="year",
    y="health_expenditure_pct_gdp",
    color="region",
    markers=True,
    title="Average Health Expenditure (% GDP) by Region"
)

fig.show()

In [120]:
df=pd.read_csv("health_indicators.csv")

plot_data = df[
    (df["year"] == latest_year)
].dropna(
    subset=[
        "health_expenditure_pct_gdp",
        "gdp_per_capita"
    ]
)

fig = px.bar(
    plot_data,
    x="gdp_per_capita",
    y="health_expenditure_pct_gdp",
    color="region",
    hover_name="country_name",
    trendline="ols",
    title="GDP per Capita vs Health Expenditure (% GDP)"
)

fig.show()

TypeError: bar() got an unexpected keyword argument 'trendline'

In [ ]:
df=pd.read_csv("health_indicators.csv")

income_avg = (
    df.groupby("income_level")
      ["life_expectancy"]
      .mean()
      .reset_index()
)

fig = px.bar(
    income_avg,
    x="income_level",
    y="life_expectancy",
    color="income_level"
)

fig.show()

In [ ]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("health_indicators.csv")

latest_year = df["year"].max()

plot_data = df[
    (df["year"] == latest_year)
].dropna(
    subset=[
        "gdp_per_capita",
        "health_expenditure_pct_gdp",
        "population"
    ]
)

fig = px.scatter(
    plot_data,
    x="gdp_per_capita",
    y="health_expenditure_pct_gdp",
    size="population",
    color="region",
    hover_name="country_name",
    size_max=50,
    title=f"GDP per Capita vs Health Expenditure (% GDP) ({latest_year})"
)

fig.update_layout(
    template="plotly_white",
    width=1000,
    height=700,
    xaxis_title="GDP per Capita",
    yaxis_title="Health Expenditure (% GDP)"
)

fig.show()

In [ ]:
top10 = plot_data.nlargest(10, "gdp_per_capita")

for _, row in top10.iterrows():

    fig.add_annotation(
        x=row["gdp_per_capita"],
        y=row["health_expenditure_pct_gdp"],
        text=row["country_name"],
        showarrow=False,
        xshift=10
    )
    fig.show()